In [2]:
pip install -q docling onnxruntime-gpu

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress t

In [16]:
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions, AcceleratorDevice, AcceleratorOptions, RapidOcrOptions
from docling.document_converter import DocumentConverter, PdfFormatOption
import torch
import json


pipeline_options = PdfPipelineOptions()
pipeline_options.do_ocr = True 
pipeline_options.do_table_structure = True 

# СОХРАНЯМ ОТДЕЛЬНО
pipeline_options.generate_table_images = True
pipeline_options.generate_picture_images = True
# 

pipeline_options.ocr_options = RapidOcrOptions(lang=["rus", "en"]) 
pipeline_options.accelerator_options = AcceleratorOptions(
    num_threads=8, 
    device=AcceleratorDevice.CUDA if torch.cuda.is_available() else AcceleratorDevice.CPU
)

converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)
    }
)

result = converter.convert("data.pdf")

with open("output.md", "w", encoding="utf-8") as f:
    f.write(result.document.export_to_markdown())

with open("output.json", "w", encoding="utf-8") as f:
    f.write(json.dumps(result.document.export_to_dict(), indent=2))

[INFO] 2026-02-22 15:58:11,464 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-02-22 15:58:11,472 [RapidOCR] download_file.py:60: File exists and is valid: /home/jupyter-semenova.ee/.local/lib/python3.12/site-packages/rapidocr/models/ch_PP-OCRv4_det_infer.onnx
[INFO] 2026-02-22 15:58:11,474 [RapidOCR] main.py:53: Using /home/jupyter-semenova.ee/.local/lib/python3.12/site-packages/rapidocr/models/ch_PP-OCRv4_det_infer.onnx
[INFO] 2026-02-22 15:58:11,690 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-02-22 15:58:11,694 [RapidOCR] download_file.py:60: File exists and is valid: /home/jupyter-semenova.ee/.local/lib/python3.12/site-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_infer.onnx
[INFO] 2026-02-22 15:58:11,695 [RapidOCR] main.py:53: Using /home/jupyter-semenova.ee/.local/lib/python3.12/site-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_infer.onnx
[INFO] 2026-02-22 15:58:11,811 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 20

In [18]:
from docling_core.types.doc import TableItem, PictureItem


doc = result.document

for element, level in doc.iterate_items():
    if isinstance(element, TableItem):
        print("Найдена таблица!")
        # Можно экспортировать её отдельно в CSV или Markdown
        print(element.export_to_dataframe())
    if isinstance(element, PictureItem) == "picture":
        print("Найден рисунок/график")

Usage of TableItem.export_to_dataframe() without `doc` argument is deprecated.
Usage of TableItem.export_to_dataframe() without `doc` argument is deprecated.
Usage of TableItem.export_to_dataframe() without `doc` argument is deprecated.
Usage of TableItem.export_to_dataframe() without `doc` argument is deprecated.
Usage of TableItem.export_to_dataframe() without `doc` argument is deprecated.
Usage of TableItem.export_to_dataframe() without `doc` argument is deprecated.
Usage of TableItem.export_to_dataframe() without `doc` argument is deprecated.
Usage of TableItem.export_to_dataframe() without `doc` argument is deprecated.
Usage of TableItem.export_to_dataframe() without `doc` argument is deprecated.
Usage of TableItem.export_to_dataframe() without `doc` argument is deprecated.
Usage of TableItem.export_to_dataframe() without `doc` argument is deprecated.
Usage of TableItem.export_to_dataframe() without `doc` argument is deprecated.
Usage of TableItem.export_to_dataframe() without `do

Найдена таблица!
                                                    0  \
0   Введение ........................................   
1                                                       
2                                                       
3                                                       
4                                                       
5                                                       
6   Глава 1. Общая информация........................   
7                                                       
8                                                       
9                                                       
10                                                      
11                                                      
12                                                      
13                                                      
14                                                      
15                                                      
16            

In [56]:
import unicodedata
import re
from typing import List, Dict, Any
from docling_core.types.doc import TableItem, PictureItem, TextItem, DocItemLabel



def clean_content_noise(text: str) -> str:
    """Очистка специфического мусора (точки, лишние пробелы)"""
    text = unicodedata.normalize('NFKC', text)
    text = re.sub(r'[\u2022\u2023\u25E6\u2043\u2219\uf0b7\uf02d\u25cf\u2713]', '-', text)
    text = re.sub(r'[\uE000-\uF8FF]', ' ', text)    
    text = re.sub(r'\.{3,}', ' ', text)
    text = re.sub(r' +', ' ', text)
    text = re.sub(r'\n\s*\|', '\n|', text)
    
    return text.strip()

def is_junk_text(text: str) -> bool:
    """Проверяет, является ли текст мусором (например, просто список цифр страниц)"""
    clean = text.replace('Раздел:', '').strip()
    if re.fullmatch(r'[\d\s\.]+', clean):
        return True
    return False

In [ ]:
import re
from typing import List, Dict, Any
from docling_core.types.doc import TableItem, PictureItem, TextItem, DocItemLabel


    
def process_docling_to_chunks(result, max_text_len: int = 500, min_merge_threshold: int = 300) -> List[Dict[str, Any]]:
    chunks = []
    text_buffer = []
    current_len = 0
    current_header = "Общая информация"
    pending_table_name = "" # Сюда будем ловить текст типа "Таблица 1..."

    for element, _ in result.document.iterate_items():
        label = element.label
        element_text = getattr(element, "text", "").strip()
        # Пропускаем оглавление оно обычно не нужно для RAG и часто содержит мусор в виде номеров страниц
        if label == DocItemLabel.DOCUMENT_INDEX or "Содержание" in element_text:
            continue

        if label in [DocItemLabel.SECTION_HEADER, DocItemLabel.TITLE]:
            # Если в буфере что-то было - сохраняем перед новым разделом
            if text_buffer:
                content = clean_content_noise(f"Раздел: {current_header}\n" + "\n".join(text_buffer))
                if not is_junk_text(content):
                    chunks.append({
                        "type": "text",
                        "content": content,
                        "metadata": {"page": getattr(element.prov[0], 'page_no', None) if element.prov else None}
                    })
                text_buffer, current_len = [], 0
            current_header = element_text
            continue

        # Если текст начинается со слова "Таблица" и он короткий
        if isinstance(element, TextItem) and re.match(r"^Таблица\s+\d+", element.text.strip()):
            pending_table_name = element.text.strip()
            continue

        if isinstance(element, TableItem):
            table_md = element.export_to_markdown()
            content = f"Таблица: {pending_table_name}\n{table_md}" if pending_table_name else table_md
        
            chunks.append({
                "type": "table",
                "content": clean_content_noise(content),
                "metadata": {
                    "context": current_header,
                    "page": getattr(element.prov[0], 'page_no', None) if element.prov else None
                }
            })
            pending_table_name = "" # Очищаем
            continue


        elif isinstance(element, PictureItem):
            # Пробуем найти подпись. Если её нет, используем pending_table_name (если там было "Рисунок...")
            caption = ""
            if hasattr(element, 'caption') and element.caption:
                caption = element.caption.text.strip()
            elif pending_table_name and "Рисунок" in pending_table_name:
                caption = pending_table_name

            # Если подпись так и не нашли - пропускаем, чтобы не плодить мусор
            if not caption:
                continue

            if text_buffer: # Сбрасываем текст перед картинкой
                chunks.append({
                    "type": "text",
                    "content": clean_content_noise(f"Раздел: {current_header}\n" + "\n".join(text_buffer)),
                    "metadata": {"context": current_header, "page": getattr(element.prov[0], 'page_no', None) if element.prov else None}
                })
                text_buffer, current_len = [], 0

            chunks.append({
                "type": "picture",
                "content": f"Раздел: {current_header}\nНайдена схема/рисунок: {caption}",
                "metadata": {"context": current_header, "page": getattr(element.prov[0], 'page_no', None) if element.prov else None}
            })
            pending_table_name = ""
            continue

            
        elif isinstance(element, TextItem):
            content = element.text.strip()
            if not content: continue
            
            text_buffer.append(content)
            current_len += len(content)

            if current_len >= max_text_len:
                content = clean_content_noise(f"Раздел: {current_header}\n" + "\n".join(text_buffer))
                if not is_junk_text(content):
                    chunks.append({
                        "type": "text",
                        "content": content,
                        "metadata": {"page": getattr(element.prov[0], 'page_no', None) if element.prov else None}
                    })
                text_buffer, current_len = [], 0
    # Хвост
    if text_buffer:
        content = clean_content_noise(f"Раздел: {current_header}\n" + "\n".join(text_buffer))
        if not is_junk_text(content):
            chunks.append({"type": "text", "content": content, "metadata": {"page": None}})

    optimized_chunks = []
    for chunk in chunks:
        if not optimized_chunks:
            optimized_chunks.append(chunk)
            continue
        
        prev = optimized_chunks[-1]
        
        # Условие склейки: предыд. чанк - текст и он слишком короткий (<300 симв)
        if prev["type"] == "text" and chunk["type"] == "text" and len(prev["content"]) < min_merge_threshold:
            # Приклеиваем текущий чанк к предыдущему
            prev["content"] += "\n\n" + chunk["content"]
            # Обновляем метаданные (страницу берем из более крупного куска)
            if not prev["metadata"]["page"]:
                prev["metadata"]["page"] = chunk["metadata"]["page"]
        else:
            optimized_chunks.append(chunk)

    return optimized_chunks

In [ ]:
chunks = process_docling_to_chunks(result)

for item in chunks:
    if item['type']=='text':
        
        print(item)

{'type': 'text', 'content': 'Раздел: ViPNet Coordinator HW 5 Подготовка к работе\nВерсия продукта: 5.3.2\nViPNet Coordinator HW50 ViPNet Coordinator HW100 ViPNet Coordinator HW1000\nViPNet Coordinator HW2000 ViPNet Coordinator HW5000 ViPNet Coordinator VA\n© АО «ИнфоТеКС», 2024 ФРКЕ.465614.003РЭ Версия продукта 5.3.2 Этот документ входит в комплект поставки продукта ViPNet, и на него распространяются все условия лицензионного соглашения. Ни одна из частей этого документа не может быть воспроизведена, опубликована, сохранена в электронной базе данных или передана в любой форме или любыми средствами, такими как электронные, механические, записывающие или иначе, для любой цели без предварительного письменного разрешения АО «ИнфоТеКС». ViPNet ® является зарегистрированным товарным знаком АО «ИнфоТеКС». Все названия компаний и продуктов, которые являются товарными знаками или зарегистрированными товарными знаками, принадлежат соответствующим владельцам. АО «ИнфоТеКС» 125167, г. Москва, вн. 

In [35]:
pip install -q "transformers==4.51.0" "sentence-transformers==5.1.1" "accelerate>=0.33.0"

Note: you may need to restart the kernel to use updated packages.


In [36]:
from sentence_transformers import SentenceTransformer, util
model =  SentenceTransformer("intfloat/multilingual-e5-large", device='cuda:2')

In [37]:
tokenizer = model.tokenizer

In [ ]:
for item in chunks:
    text = item['content']
    token_ids = tokenizer.encode(text)
    if len(token_ids)<100:
        print(item)

Usage of TableItem.export_to_markdown() without `doc` argument is deprecated.
Usage of TableItem.export_to_markdown() without `doc` argument is deprecated.
Usage of TableItem.export_to_markdown() without `doc` argument is deprecated.
Usage of TableItem.export_to_markdown() without `doc` argument is deprecated.
Usage of TableItem.export_to_markdown() without `doc` argument is deprecated.
Usage of TableItem.export_to_markdown() without `doc` argument is deprecated.
Usage of TableItem.export_to_markdown() without `doc` argument is deprecated.
Usage of TableItem.export_to_markdown() without `doc` argument is deprecated.
Usage of TableItem.export_to_markdown() without `doc` argument is deprecated.
Usage of TableItem.export_to_markdown() without `doc` argument is deprecated.
Usage of TableItem.export_to_markdown() without `doc` argument is deprecated.
Usage of TableItem.export_to_markdown() without `doc` argument is deprecated.
Usage of TableItem.export_to_markdown() without `doc` argument i

{'type': 'text', 'content': 'Раздел: О документе\n- Настройка с помощью командного интерпретатора.\n- Настройка с помощью веб-интерфейса.\n- Справочник команд и конфигурационных файлов.\n- Лицензионные соглашения на компоненты сторонних производителей.\n- История версий.\n- Перечень совместимых трансиверов.', 'metadata': {'page': 8}}
{'type': 'text', 'content': 'Раздел: Транспортный сервер MFTP\nViPNet-клиенты и ViPNet Coordinator HW версий ниже 5.0 обмениваются с ViPNet Prime транспортными конвертами MFTP. Если обмен настроен через ViPNet Coordinator HW, то для передачи конвертов используется транспортный сервер MFTP ViPNet Coordinator HW, работающий в транзитном режиме.', 'metadata': {'page': 26}}
{'type': 'text', 'content': 'Раздел: ViPNet Coordinator HW100\nИсполнение ViPNet Coordinator HW100 применяется для защиты небольших офисов и удаленных рабочих мест. Исполнение распространяется на аппаратных платформах HW100 N1, HW100 N2, HW100 N3, HW100 Q1 и HW100 Q2.', 'metadata': {'page':

{'type': 'text', 'content': 'Раздел: Введение\n7\n8\n10\n11\n12', 'metadata': {'page': 7}}
{'type': 'text', 'content': 'Раздел: Соглашения документа\nВнимание! Все сценарии работы с ViPNet Coordinator HW приведены для локального администратора admin в режимах просмотра или настройки.', 'metadata': {'page': 8}}
{'type': 'text', 'content': 'Раздел: Описание команд:\nhostname# iplir ping 0x270e000a', 'metadata': {'page': 10}}
{'type': 'text', 'content': 'Раздел: Что нового в версии 5.3.2\nПовышена стабильность работы ViPNet Coordinator HW. Исправлены ошибки, обнаруженные при эксплуатации предыдущих версий ViPNet Coordinator HW.', 'metadata': {'page': 11}}
{'type': 'text', 'content': 'Раздел: Средство предотвращения вторжений (IPS)\n- Попытки эксплуатации уязвимостей в ПО объектов защищаемой сети.\n- Атаки на сетевые службы и серверы.\n- Атаки типа «отказ в обслуживании» (DoS-атаки).\n- Аномальный IP-трафик.\n- Сетевую активность вирусов.', 'metadata': {'page': 18}}
{'type': 'text', 'conte

In [ ]:
def create_embeddings(data_list, model, batch_size=32):
    """
    Создает эмбеддинги для списка чанков.
    Использует GPU для пакетной обработки (batch processing).
    """
    if not data_list:
        print("Список данных пуст.")
        return []

    print(f"Подготовка текстов для {len(data_list)} чанков...")
    
    # Подготавливаем тексты с префиксом "passage: " (нужно для моделей E5)
    # Используем context из метаданных как заголовок для эмбеддинга
    prepared_texts = []
    for item in data_list:
        context = item.get("metadata", {}).get("context", "Общая информация")
        text = item.get("content", "")
        prepared_texts.append(f"passage: {context}. {text}")

    print(f"Начинаем расчет векторов на {model.device} (batch_size={batch_size})...")
    
    all_vectors = model.encode(
        prepared_texts,
        batch_size=batch_size,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True
    )

    final_list = []
    for i, item in enumerate(data_list):
        new_item = item.copy()
        new_item["vectors"] = all_vectors[i].tolist()
        final_list.append(new_item)

    print("Все эмбеддинги созданы успешно.")
    return final_list

In [74]:
chunks_with_vectors = create_embeddings(chunks, model)

Подготовка текстов для 156 чанков...
Начинаем расчет векторов на cuda:2 (batch_size=32)...


Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Все эмбеддинги созданы успешно.
Размер вектора первого чанка: 1024


In [75]:
%pip install chromadb

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.5/21.5 MB 25.3 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.7/6.7 MB 20.9 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 39.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 32.3 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.4/4.4 MB 32.7 MB/s eta 0:00:00
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH o

In [ ]:
import chromadb
from chromadb.config import Settings


def save_to_chroma(chunks_with_vectors, collection_name="vipnet_docs"):
    """
    Создает локальную базу данных и загружает в нее чанки с готовыми векторами.
    """
    # Инициализируем клиента Chroma
    client = chromadb.PersistentClient(path="./chroma_db")
    

    # отключаем встроенную модель эмбеддингов, так как используем свою
    collection = client.get_or_create_collection(name=collection_name)

    ids = []
    documents = []
    metadatas = []
    embeddings = []

    for i, item in enumerate(chunks_with_vectors):
        ids.append(f"id_{i}")
        documents.append(item["content"])
        # Метаданные (нужно превратить None в пустые строки или числа, Chroma не любит None)
        meta = {
            "page": item["metadata"].get("page") if item["metadata"].get("page") else 0,
            "context": item["metadata"].get("context", ""),
            "type": item["type"]
        }
        metadatas.append(meta)
        embeddings.append(item["vectors"])

    # Загружаем всё в базу
    collection.upsert(
        ids=ids,
        metadatas=metadatas,
        documents=documents,
        embeddings=embeddings
    )
    
    print(f"База данных готова! Загружено {len(ids)} чанков.")
    return collection


def semantic_search(query, collection, model, top_k=3):

    prepared_query = f"query: {query}"
    
    query_vector = model.encode(
        [prepared_query], 
        normalize_embeddings=True,
        show_progress_bar=False
    ).tolist()

    results = collection.query(
        query_embeddings=query_vector,
        n_results=top_k,
        include=["documents", "metadatas", "distances"]
    )

    # 4. Красиво форматируем вывод
    formatted_results = []
    for i in range(len(results["ids"][0])):
        formatted_results.append({
            "content": results["documents"][0][i],
            "page": results["metadatas"][0][i]["page"],
            "context": results["metadatas"][0][i]["context"],
            "score": round(1 - results["distances"][0][i], 4) # Чем ближе к 1, тем точнее
        })
    
    return formatted_results

In [77]:
collection = save_to_chroma(chunks_with_vectors)


База данных готова! Загружено 156 чанков.


In [78]:
question = "Какие характеристики у ViPNet Coordinator HW1000?"
results = semantic_search(question, collection, model, top_k=3)

print(f"Вопрос: {question}\n")
for i, res in enumerate(results):
    print(f"--- Результат #{i+1} (Сходство: {res['score']}) ---")
    print(f"Раздел: {res['context']} | Страница: {res['page']}")
    print(f"Текст: {res['content'][:400]}...")
    print("-" * 50)

Вопрос: Какие характеристики у ViPNet Coordinator HW1000?

--- Результат #1 (Сходство: 0.8069) ---
Раздел:  | Страница: 32
Текст: Раздел: ViPNet Coordinator HW1000
Исполнения ViPNet Coordinator HW1000 могут быть использованы для защиты компьютерных сетей масштаба предприятия....
--------------------------------------------------
--- Результат #2 (Сходство: 0.7905) ---
Раздел:  | Страница: 39
Текст: Раздел: ViPNet Coordinator HW5000
Исполнение ViPNet Coordinator HW5000 может быть использовано для защиты магистральных каналов связи, организации защищенного доступа в центры обработки данных и к облачным ресурсам. Исполнение распространяется на аппаратных платформах HW5000 Q1 и HW5000 Q2....
--------------------------------------------------
--- Результат #3 (Сходство: 0.7876) ---
Раздел: ViPNet Coordinator HW1000 | Страница: 32
Текст: Таблица: Таблица 5. Характеристики HW100 Q1, Q2
| Исполнение | Аппаратные платформы |
|-----------------------------|------------------------|
| ViPNet Coor